# rates-vol-rv: results

Everything here reads `results/run.json` and `data/derived/*.parquet` written by `python -m ratesvol run`; re-run the pipeline and re-execute to refresh.

In [ ]:
import json, os, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))
from ratesvol import data as D, cube as C, events as E, vrp as V, rv as R, density as Z
R_ = json.load(open('../results/run.json'))
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

## 1. The bp-vol cube

ATM normal vol by fund (the tenor axis) and expiry, from the recorded closing chains; MOVE for reference.  Skew is the bp vol 50 bp above the forward yield minus 50 bp below.

In [ ]:
funds = pd.DataFrame(R_['cube']['funds']).T
funds[['label','duration','n_expiries','n_quotes','atm_bpvol_1m','atm_bpvol_3m','atm_bpvol_6m','skew_50bp_1m','rmse_bpvol_median','inside_market','atm_halfspread_vol_1m']]

In [ ]:
from IPython.display import Image, display
display(Image('../results/figures/cube.png')); display(Image('../results/figures/smile_fit.png'))

## 2. Event variance

Left: the implied one-day move of each scheduled event from the TLT expiry ladder (NNLS on total variance).  Right: 22 years of realised over index-implied one-day moves by event type.

In [ ]:
pd.DataFrame(R_.get('event_extraction', []))

In [ ]:
display(Image('../results/figures/events.png'))
for k, h in R_['event_history'].items():
    print(k, h['from'], h['to']); display(pd.DataFrame(h['by_event']).set_index('kind').round(3))

## 3. The variance risk premium and the forecast test

In [ ]:
display(pd.DataFrame(R_['vrp']['summary']).set_index('index').round(3))
display(pd.DataFrame(R_['vrp']['forecast']).set_index('index').round(3))
display(Image('../results/figures/vrp.png'))

## 4. Strategies on the taking side, net of measured costs

In [ ]:
print(R_['strategies']['costs'])
display(pd.DataFrame(R_['strategies']['strategies']).set_index('strategy').round(3))
display(pd.DataFrame(R_['strategies']['event_strategies']).set_index('strategy').round(3))
display(Image('../results/figures/strategies.png'))

## 5. Taking costs, densities and the policy distribution

In [ ]:
display(Image('../results/figures/taking_cost.png')); display(Image('../results/figures/density.png'))
if 'mpt' in R_: print({k: v for k, v in R_['mpt'].items() if k != 'table'})

## 6. SQL over the store

The DuckDB views cover the reference tables and every derived parquet.

In [ ]:
con = D.store()
con.execute("SELECT symbol, tenor_bucket, money_bucket, n, halfspread_bpvol FROM taking_costs WHERE symbol='TLT' ORDER BY tenor_bucket, money_bucket").df()